# سعّر — خادم صورة واحدة على Kaggle

هذه نسخة مستقلة مبنية على الدفتر المرفق. لا تعدل الدفتر القديم. شغل هذا الدفتر في جلسة Kaggle جديدة مع نفس Dataset والأسرار المسماة سابقًا. لا تشغل الخادم القديم والجديد معًا على نطاق ngrok نفسه. إذا كانت الجلسة القديمة متوقفة، لا يلزم إعادة تشغيلها.

شغل الخلايا من الأعلى للأسفل. خلية إصلاح العقد تأتي بعد فحص الأصول وقبل استيراد app؛ تحفظ app السابق ولا تعدل الأوزان. أزيلت خلية تثبيت Gradio 5.49.1 المتعارضة مع خلية المتطلبات التي تحدد 6.31.0. خلية فحص العقد تثبت صورة واحدة لكل من السعر والهوية. اختبار الصورة الأخير اختياري؛ لا يعد التنبؤ ناجحًا دون تشغيله وظهور نتيجة فعلية، ثم اختبار الموقع. لا توجد نتائج محفوظة في الدفتر.


In [ ]:
# 1) العثور على الحزمة ونسخها إلى مكان قابل للكتابة
from pathlib import Path
import shutil, zipfile

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/saer-backend-bundle')
if not INPUT_ROOT.is_dir():
    raise RuntimeError('لم أجد /kaggle/input؛ أضف Dataset الخاص إلى هذا الدفتر أولاً.')

src_root = next((p.parent for p in sorted(INPUT_ROOT.rglob('app.py')) if (p.parent / 'model_store.py').is_file()), None)
if src_root is not None:
    shutil.copytree(src_root, WORK_ROOT, dirs_exist_ok=True)
else:
    archives = sorted(INPUT_ROOT.rglob('saer-backend-bundle.zip')) or sorted(INPUT_ROOT.rglob('*.zip'))
    if len(archives) != 1:
        raise RuntimeError(f'توقعت ملف حزمة ZIP واحداً؛ وجدت: {len(archives)}')
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(WORK_ROOT)

bundle_root = WORK_ROOT
if not (bundle_root / 'requirements.txt').is_file():
    raise RuntimeError(f'requirements.txt غير موجود في {bundle_root}')
print(f'مسار الحزمة: {bundle_root}')
print('عدد الملفات:', sum(1 for p in bundle_root.rglob('*') if p.is_file()))

In [ ]:
# الإصدارات المثبتة؛ خلية المتطلبات التالية هي المسؤولة عن التثبيت.
import importlib.metadata as md
for package in ('gradio', 'gradio-client'):
    try:
        print(package, md.version(package))
    except md.PackageNotFoundError:
        print(package, 'غير مثبت بعد')


In [ ]:
# 2) المتطلبات: تثبيت مرتب وآمن (يتخطى ما هو مطابق)
import importlib.metadata as md, subprocess, sys
from packaging.requirements import Requirement
from packaging.version import Version

def satisfied(req):
    try:
        v = Version(md.version(req.name))
    except md.PackageNotFoundError:
        return False
    return (not req.specifier) or req.specifier.contains(v, prereleases=True)

def pip_install(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-conflicts', *args], check=True)

reqs = []
for raw in (bundle_root / 'requirements.txt').read_text(encoding='utf-8').splitlines():
    raw = raw.strip()
    if raw and not raw.startswith('#'):
        reqs.append(Requirement(raw))

extras = [Requirement('gradio==6.31.0'), Requirement('pyngrok>=9.1.1,<10')]
# الترتيب مهم: gradio أولاً لأنه يرفع huggingface_hub، ثم متطلبات الحزمة، ثم تثبيت نهائي لما تغيّر
for group in (extras, reqs, reqs):
    todo = [str(r) for r in group if not satisfied(r)]
    if todo:
        print('تثبيت:', ', '.join(todo))
        pip_install(*todo)

still = [str(r) for r in reqs + extras if not satisfied(r)]
if still:
    raise RuntimeError('تعذر ضبط هذه الإصدارات: ' + ', '.join(still))

# هل هناك حزمة محمّلة في الذاكرة بإصدار مختلف عن المثبّت؟ (يحتاج Restart)
mods = {'scikit-learn': 'sklearn'}
stale = []
for r in reqs:
    name = mods.get(r.name, r.name.replace('-', '_'))
    m = sys.modules.get(name)
    if m is not None and getattr(m, '__version__', None):
        if Version(m.__version__).public != Version(md.version(r.name)).public:
            stale.append(f'{r.name}: محمّل {m.__version__} / مثبّت {md.version(r.name)}')
if stale:
    raise RuntimeError('أعد تشغيل الجلسة (Run ← Restart Session) ثم اختر Run All. السبب:\n' + '\n'.join(stale))

# اختبار الاستيراد في عملية نظيفة
test = subprocess.run([sys.executable, '-c', 'import numpy, pandas, sklearn, torch, torchvision, transformers, huggingface_hub, gradio, fastapi, uvicorn'], capture_output=True, text=True)
if test.returncode != 0:
    raise RuntimeError('فشل استيراد الحزم:\n' + test.stderr[-1500:])
print('كل المتطلبات جاهزة. huggingface_hub =', md.version('huggingface_hub'), '| transformers =', md.version('transformers'), '| numpy =', md.version('numpy'))

In [ ]:
# 3) فحص الأصول وإعداد البيئة
import hashlib, json, os, sys
from urllib.parse import urlsplit

manifest = json.loads((bundle_root / 'private_assets_manifest.json').read_text(encoding='utf-8'))
problems = []
for item in manifest['files']:
    path = bundle_root / item['path']
    if not path.is_file():
        problems.append(f"مفقود: {item['path']}")
        continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if path.stat().st_size != item['size_bytes'] or digest != item['sha256']:
        problems.append(f"فشل الحجم أو SHA-256: {item['path']}")
if problems:
    raise RuntimeError('\n'.join(problems))
print(f"فحصت الأصول: {manifest['total_bytes']:,} بايت؛ كل البصمات مطابقة.")

PAGES_PROJECT_URL = 'https://saer-rmsavg637.pages.dev'  # استبدله برابط Pages الفعلي
parsed_pages = urlsplit(PAGES_PROJECT_URL)
PAGES_ORIGIN = f'{parsed_pages.scheme}://{parsed_pages.netloc}'
os.environ['SAER_APP_ROOT'] = str(bundle_root)
os.environ['LOCAL_ASSETS_DIR'] = str(bundle_root)
os.environ['SAER_RUNTIME_CACHE'] = '/kaggle/working/saer-runtime-cache'
os.environ['FRONTEND_ORIGINS'] = PAGES_ORIGIN
sys.path.insert(0, str(bundle_root))
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# إصلاح العقد قبل تشغيل الخادم؛ لا يستدعي النماذج ولا ينزل الأوزان.
import sys
if 'app' in sys.modules:
    raise RuntimeError('app محمل بالفعل؛ شغل هذا الدفتر في جلسة جديدة قبل تشغيل الخادم.')
"""إصلاح عقد الصورة في حزمة Kaggle دون تغيير الأوزان أو تنزيل ملفات."""

import ast
from datetime import datetime, timezone
from pathlib import Path


def repair(root: Path) -> Path:
    path = root / "app.py"
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)
    functions = {node.name: node for node in tree.body if isinstance(node, ast.FunctionDef)}
    for name in ("_valid_image_path", "estimate_image_price", "identify_vehicle"):
        if name not in functions:
            raise RuntimeError(f"توقف: لم أجد الدالة المتوقعة {name}؛ لم أعدل الملف")
    replacements = {
        "_valid_image_path": '''def _valid_image_path(value):
    if isinstance(value, (list, tuple)):
        raise ValueError("ارفع صورة واحدة فقط للسيارة")
    if not isinstance(value, (str, Path)) or not value:
        raise ValueError("اختر صورة السيارة أولًا")
    if _startup_error:
        raise RuntimeError(_startup_error)
    path = Path(value)
    if not path.is_file() or not 0 < path.stat().st_size <= 12 * 1024 * 1024:
        raise ValueError("الصورة مفقودة أو فارغة أو حجمها يتجاوز 12 ميجابايت")
    from PIL import Image, UnidentifiedImageError
    try:
        with Image.open(path) as image:
            if image.format not in {"JPEG", "PNG", "WEBP"}:
                raise ValueError("الصيغ المدعومة: JPG وPNG وWebP")
            if image.width * image.height > 40_000_000:
                raise ValueError("أبعاد الصورة كبيرة جداً")
            image.verify()
    except (OSError, UnidentifiedImageError, Image.DecompressionBombError) as exc:
        raise ValueError("ملف الصورة غير صالح") from exc
    return path
''',
        "estimate_image_price": '''def estimate_image_price(upload):
    paths = upload if isinstance(upload, list) else [upload]
    if len(paths) != 1:
        raise ValueError("ارفع صورة واحدة فقط للسيارة")
    checked = [_valid_image_path(path) for path in paths]
    digest = hashlib.sha256("|".join(services.file_sha256(path) for path in checked).encode()).hexdigest()
    result = services.predict_image_price(checked, digest)
    notice = "تقدير تجريبي من صورة واحدة؛ لا يرى الممشى أو الحالة وليس سعر بيع مضموناً."
    return {**result, "image_count": 1, "notice_ar": notice, "notice": notice}
''',
    }
    lines = source.splitlines(keepends=True)
    edits = [(functions[name].lineno - 1, functions[name].end_lineno, code + "\n") for name, code in replacements.items()]
    found = set()
    for node in ast.walk(tree):
        if not isinstance(node, ast.Assign) or not isinstance(node.value, ast.Call):
            continue
        names = [target.id for target in node.targets if isinstance(target, ast.Name)]
        for name in ("identity_file", "price_files"):
            if name not in names:
                continue
            if ast.unparse(node.value.func) != "gr.File":
                raise RuntimeError(f"توقف: {name} ليس gr.File")
            found.add(name)
            node.value.keywords = [kw for kw in node.value.keywords if kw.arg not in {"file_count", "label"}]
            node.value.keywords.extend([ast.keyword(arg="file_count", value=ast.Constant("single")), ast.keyword(arg="label", value=ast.Constant("صورة السيارة"))])
            indent = " " * node.col_offset
            edits.append((node.lineno - 1, node.end_lineno, indent + ast.unparse(node) + "\n"))
    if found != {"identity_file", "price_files"}:
        raise RuntimeError("توقف: لم أجد مدخلي Gradio المتوقعين؛ لم أعدل الملف")
    for start, end, code in sorted(edits, reverse=True):
        lines[start:end] = [code]
    updated = "".join(lines)
    compile(updated, str(path), "exec")
    backup = path.with_name("app.before-single-image-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f") + ".py")
    backup.write_bytes(path.read_bytes())
    path.write_text(updated, encoding="utf-8")
    print("تم إصلاح مدخلي الصورة، وحفظ النسخة السابقة:", backup)
    return backup


repair(bundle_root)


In [ ]:
# 4) تشغيل الخادم والنفق
import json, logging, os, queue, re, subprocess, sys, threading, time, urllib.request
from pathlib import Path
from urllib.parse import quote
import requests

def kaggle_secret(name):
    value = os.environ.get(name, '')
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name) or value
    except Exception:
        pass
    return value.strip() if isinstance(value, str) else ''

NGROK_AUTHTOKEN = kaggle_secret('NGROK_AUTHTOKEN')
NGROK_STATIC_DOMAIN = kaggle_secret('NGROK_STATIC_DOMAIN')
GITHUB_GIST_TOKEN = kaggle_secret('GITHUB_GIST_TOKEN')
GIST_ID = kaggle_secret('GIST_ID')
TUNNEL_MODE = 'auto'  # auto أو ngrok أو gradio أو cloudflared
mode = TUNNEL_MODE
if mode == 'auto':
    mode = 'ngrok' if NGROK_AUTHTOKEN else 'gradio'
if mode not in ('ngrok', 'gradio', 'cloudflared'):
    raise ValueError('TUNNEL_MODE يجب أن يكون auto أو ngrok أو gradio أو cloudflared.')

import app as saer_server
if saer_server._startup_error:
    raise RuntimeError(saer_server._startup_error)

def start_uvicorn():
    if globals().get('_uvicorn_started'):
        return
    globals()['_uvicorn_started'] = True
    import uvicorn
    threading.Thread(target=lambda: uvicorn.run(saer_server.app, host='0.0.0.0', port=7860, log_level='warning'), daemon=True).start()
    for _ in range(90):
        try:
            urllib.request.urlopen('http://127.0.0.1:7860/config', timeout=2).read(64)
            return
        except Exception:
            time.sleep(1)
    raise RuntimeError('لم يبدأ خادم Uvicorn على المنفذ 7860.')

api_url = None
if mode == 'ngrok':
    if not NGROK_AUTHTOKEN:
        raise RuntimeError('أضف NGROK_AUTHTOKEN إلى Kaggle Secrets أو اضبط TUNNEL_MODE على gradio.')
    from pyngrok import conf, ngrok
    logging.getLogger('pyngrok').setLevel(logging.WARNING)
    Path('/kaggle/temp').mkdir(parents=True, exist_ok=True)
    config_path = '/kaggle/temp/saer-ngrok.yml'
    try:
        ngrok.kill()
    except Exception:
        pass
    Path(config_path).unlink(missing_ok=True)
    ngrok_config = conf.PyngrokConfig(config_path=config_path)
    ngrok.set_auth_token(NGROK_AUTHTOKEN, pyngrok_config=ngrok_config)
    start_uvicorn()
    clean_domain = NGROK_STATIC_DOMAIN.removeprefix('https://').removeprefix('http://').split('/', 1)[0]
    if clean_domain:
        try:
            tunnel = ngrok.connect(addr=7860, proto='http', url=f'https://{clean_domain}', pyngrok_config=ngrok_config)
        except Exception as e:
            print('تعذر استخدام النطاق الثابت:', str(e)[:300])
            print('سأستخدم نطاقاً عشوائياً بدلاً منه.')
            tunnel = ngrok.connect(addr=7860, proto='http', pyngrok_config=ngrok_config)
    else:
        tunnel = ngrok.connect(addr=7860, proto='http', pyngrok_config=ngrok_config)
    api_url = tunnel.public_url
    Path(config_path).unlink(missing_ok=True)
elif mode == 'gradio':
    result = saer_server.launch_gradio_share(prevent_thread_lock=True)
    api_url = next((v for v in result if isinstance(v, str) and '.gradio.live' in v), None) if isinstance(result, (tuple, list)) else None
    api_url = api_url or getattr(saer_server.demo, 'share_url', None)
elif mode == 'cloudflared':
    start_uvicorn()
    cloudflared = '/kaggle/working/cloudflared'
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    os.chmod(cloudflared, 0o755)
    proc = subprocess.Popen([cloudflared, 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    threading.Thread(target=lambda: [lines.put(line) for line in proc.stdout], daemon=True).start()
    deadline = time.time() + 90
    while time.time() < deadline and not api_url:
        try:
            line = lines.get(timeout=1)
        except queue.Empty:
            continue
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            api_url = match.group(0)

if not api_url:
    raise RuntimeError('لم يُستخرج رابط HTTPS عام؛ راجع الإنترنت وإعداد النفق.')
api_url = api_url.rstrip('/')
print('رابط API العام:')
print(api_url)

gist_config_url = ''
if GITHUB_GIST_TOKEN and GIST_ID:
    payload = {'api_base_url': api_url, 'updated_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
    headers = {'Authorization': f'Bearer {GITHUB_GIST_TOKEN}', 'Accept': 'application/vnd.github+json', 'X-GitHub-Api-Version': '2022-11-28'}
    response = requests.patch(f'https://api.github.com/gists/{GIST_ID}', headers=headers, json={'files': {'saer-api.json': {'content': json.dumps(payload, ensure_ascii=False)}}}, timeout=30)
    if not response.ok:
        raise RuntimeError(f'فشل تحديث Gist (HTTP {response.status_code}).')
    gist_config_url = f'https://api.github.com/gists/{GIST_ID}'
    print('تم تحديث ملف الإعداد في Gist.')

PAGES_PROJECT_URL = globals().get('PAGES_PROJECT_URL', 'https://saer-rmsavg637.pages.dev')
base = PAGES_PROJECT_URL.rstrip('/')
if gist_config_url:
    frontend_url = f'{base}/?apiConfig={quote(gist_config_url, safe="")}'
else:
    frontend_url = f'{base}/?api={quote(api_url, safe="")}'
print('\n' + '=' * 72)
print('الرابط النهائي لواجهة Cloudflare Pages:')
print(frontend_url)
print('=' * 72)
print('لا توقف جلسة Kaggle؛ الخادم يتوقف بتوقفها.')

In [ ]:
# فحص العقد المحلي والعام بعد تشغيل الخادم والنفق.
import requests
headers = {'ngrok-skip-browser-warning': 'true'}
for base in ('http://127.0.0.1:7860', api_url):
    response = requests.get(base.rstrip('/') + '/config', headers=headers, timeout=45)
    response.raise_for_status()
    cfg = response.json()
    components = {item['id']: item for item in cfg['components']}
    for endpoint in ('predict_image', 'identify_vehicle'):
        dep = next(item for item in cfg['dependencies'] if item.get('api_name') == endpoint)
        assert len(dep['inputs']) == 1, (endpoint, 'عدد معاملات غير متوقع')
        comp = components[dep['inputs'][0]]
        assert comp['type'] == 'file' and comp['props']['file_count'] == 'single', endpoint
        assert comp['api_info']['type'] == 'object', endpoint
        print(base, endpoint, 'FileData واحد — صحيح')
# فحص الرفض في الدالة نفسها قبل تحميل/استدعاء النموذج.
try:
    saer_server.estimate_image_price(['first.jpg', 'second.jpg'])
except ValueError as exc:
    assert 'صورة واحدة' in str(exc)
    print('رفض أكثر من صورة:', exc)
else:
    raise AssertionError('الخادم لم يرفض الصور المتعددة')
print('نجح فحص العقد. هذا ليس إثباتًا لنجاح التنبؤ؛ شغل اختبار الصورة ثم الموقع.')


## اختبار حقيقي اختياري
حدد مسار صورة سيارة فعلية في TEST_IMAGE. الخلية ترفع صورة واحدة لكل مدخل وترفض السعر غير الصالح. لا تستخدم نتائج الخلية بدل اختبار ظهور النتيجة في الموقع. يبين الحقل «من_ذاكرة_مؤقتة» إن كان الخادم أعاد نتيجة من الكاش. لا تُشارك الصورة إلا إذا تملك صلاحية استخدامها.


In [ ]:
TEST_IMAGE = ""  # مثال: /kaggle/input/your-dataset/car.jpg
if not TEST_IMAGE:
    print('لم يُختبر التنبؤ بعد: ضع مسار صورة فعلية ثم شغل هذه الخلية.')
else:
    from pathlib import Path
    from gradio_client import Client, handle_file
    import math
    image = Path(TEST_IMAGE)
    assert image.is_file(), 'لم أجد الصورة'
    client = Client(api_url, headers={'ngrok-skip-browser-warning': 'true'})
    price = client.predict(handle_file(str(image)), api_name='/predict_image')
    assert isinstance(price, dict) and isinstance(price.get('price_sar'), (int, float)) and math.isfinite(price['price_sar']), price
    assert price.get('image_count') == 1, price
    print('نتيجة السعر الفعلية:', price)
    identity = client.predict(handle_file(str(image)), api_name='/identify_vehicle')
    assert isinstance(identity, dict) and identity.get('الاقتراح_الرئيسي'), identity
    print('نتيجة التعرف الفعلية:', identity)
